# P4 — Enterprise Customer Support Intent Router
Build a ticket router with structured outputs and compare prompt-style baselines. No API key is required for the baseline; an LLM API integration is an optional extension.

In [30]:
!pip -q install scikit-learn pydantic

In [31]:
from pathlib import Path
import os
DATA_DIR = Path("/kaggle/input/datasets/babblu123/real-world-ml-genai-kaggle-complete/real_world_ml_genai_lab/datasets")
if not DATA_DIR.exists():
    DATA_DIR = Path("../datasets")
if not DATA_DIR.exists():
    DATA_DIR = Path("datasets")
print("Dataset directory:", DATA_DIR.resolve())

import pandas as pd, json, re
df=pd.read_csv(DATA_DIR/"p4_support_tickets.csv")
df.head()

Dataset directory: /kaggle/input/datasets/babblu123/real-world-ml-genai-kaggle-complete/real_world_ml_genai_lab/datasets


,ticket_id,email_text,category,priority
0,T00001,Invoice charged twice on my account; details r...,billing,low
1,T00002,The API returns HTTP 500 when I upload a file,technical_api,medium
2,T00003,Please add dark mode to the dashboard,feature_request,low
3,T00004,I cannot reset my password; details reference ...,account_access,medium
4,T00005,My monthly subscription was renewed unexpectedly,billing,high


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
Xtr,Xte,ytr,yte=train_test_split(df.email_text,df.category,test_size=.2,random_state=42,stratify=df.category)
router=Pipeline([("tfidf",TfidfVectorizer(ngram_range=(1,2))),("clf",LogisticRegression(max_iter=1000))])
router.fit(Xtr,ytr); pred=router.predict(Xte)
print(classification_report(yte,pred))

                 precision    recall  f1-score   support

 account_access       1.00      1.00      1.00        30
        billing       1.00      1.00      1.00        30
feature_request       1.00      1.00      1.00        30
  technical_api       1.00      1.00      1.00        30

       accuracy                           1.00       120
      macro avg       1.00      1.00      1.00       120
   weighted avg       1.00      1.00      1.00       120



In [33]:
from pydantic import BaseModel, Field
from typing import Literal
class TicketRoute(BaseModel):
    category: Literal["billing","technical_api","feature_request","account_access"]
    priority: Literal["low","medium","high"]
    draft_response: str = Field(min_length=10,max_length=500)
def route_ticket(text):
    category=router.predict([text])[0]
    priority="high" if any(w in text.lower() for w in ["charged twice","security","cannot login","500","timeout"]) else "medium"
    return TicketRoute(category=category,priority=priority,draft_response=f"Thanks for contacting support. Your {category.replace('_',' ')} request has been recorded; our team will review it.")
sample=route_ticket("The API returns HTTP 500 when I upload a file")
print(sample.model_dump_json(indent=2))

{
  "category": "technical_api",
  "priority": "high",
  "draft_response": "Thanks for contacting support. Your technical api request has been recorded; our team will review it."
}


## Optional LLM extension
Add an API key through Kaggle Secrets (never hard-code it). Require JSON-schema output, implement retries/backoff, validate with Pydantic, and benchmark accuracy/latency against the baseline. Do not expose chain-of-thought; record concise rationale only.